# Expanded Taxonomy Decision Layer (E2)



## 1. Purpose

- Convert E2 expanded-taxonomy predictions into product decision bands.
- Keep policy generation reproducible from Kaggle outputs.
- Export lightweight artifacts required by the backend/demo contract.



In [ ]:
from dataclasses import dataclass

from pathlib import Path

import zipfile
import json


import numpy as np

import pandas as pd



In [ ]:
@dataclass(frozen=True)
class CFG:
    """Configuration for E2 expanded-taxonomy decision-layer analysis."""

    INPUT_DIR: Path = Path(
        "/kaggle/working/results/expanded_taxonomy/e2_expanded_taxonomy_v2_finetune_224"
    )
    NOTEBOOK_OUTPUT_DIR: Path = Path("/kaggle/input/notebooks")
    ATTACHMENT_ROOT: Path = Path("/kaggle/input")
    RESULTS_DIR: Path = Path("/kaggle/working/results/expanded_taxonomy_v2_decision_layer")
    ZIP_EXTRACT_DIR: Path = Path(
        "/kaggle/working/results/expanded_taxonomy_v2_decision_layer_from_zip"
    )
    ZIP_FILE_NAME: str = "expanded_taxonomy_v2_decision_layer_artifacts.zip"

    PREDICTION_CANDIDATES: tuple[str, ...] = (
        "test_predictions.csv",
        "test_predictions_calibrated.csv",
    )
    HARD_CLASS_CANDIDATES: tuple[str, ...] = (
        "weak_class_report.csv",
        "hard_classes_calibrated.csv",
    )
    CONFUSION_CANDIDATES: tuple[str, ...] = (
        "top_confusion_pairs.csv",
        "top_confusion_pairs_calibrated.csv",
    )
    METRICS_FILE: str = "expanded_metrics.csv"

    MIN_AUTO_ACCEPT_ACCURACY: float = 0.90
    MIN_SUGGEST_ACCURACY: float = 0.80

    AUTO_CONFIDENCE_GRID: tuple[float, ...] = tuple(np.round(np.arange(0.70, 0.96, 0.05), 2))
    SUGGEST_CONFIDENCE_GRID: tuple[float, ...] = tuple(np.round(np.arange(0.35, 0.76, 0.05), 2))
    MARGIN_GRID: tuple[float, ...] = (0.0,)

    @staticmethod
    def input_roots() -> list[Path]:
        roots = [
            CFG.INPUT_DIR,
            CFG.NOTEBOOK_OUTPUT_DIR,
            CFG.ATTACHMENT_ROOT,
            Path("/kaggle/input/datasets"),
            CFG.ZIP_EXTRACT_DIR,
        ]
        roots.extend(sorted(CFG.NOTEBOOK_OUTPUT_DIR.glob("**/results/expanded_taxonomy/e2_expanded_taxonomy_v2_finetune_224")))
        return list(dict.fromkeys(roots))

    @staticmethod
    def ensure_dirs() -> None:
        CFG.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        CFG.ZIP_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)



## 2. Load Outputs

Resolve prediction logs, weak-class summary, and conflict pair candidates from the
attached E2 notebook output.

If a compact zip artifact is attached, it is extracted automatically.



In [ ]:
CFG.ensure_dirs()

def extract_artifact_zip() -> None:
    """Extract bundled decision-layer artifacts if provided."""
    zip_candidates = []
    for root in CFG.input_roots():
        if root.exists():
            zip_candidates.extend(sorted(root.rglob(CFG.ZIP_FILE_NAME)))

    if not zip_candidates:
        return

    zip_path = zip_candidates[0]
    marker_file = CFG.ZIP_EXTRACT_DIR / ".extracted"
    if marker_file.exists():
        return

    with zipfile.ZipFile(zip_path, "r") as artifact_zip:
        artifact_zip.extractall(CFG.ZIP_EXTRACT_DIR)
    marker_file.write_text(str(zip_path))
    print(f"Extracted artifact zip: {zip_path}")


def resolve_file(candidates: tuple[str, ...], required: bool = True) -> Path | None:
    """Resolve a file path from likely Kaggle attachment roots."""
    extract_artifact_zip()

    matches = []
    for root in CFG.input_roots():
        for candidate in candidates:
            direct = root / candidate
            if direct.exists():
                return direct
            matches.extend(sorted(root.rglob(candidate)))
        if matches:
            return matches[0]

    if required:
        raise FileNotFoundError(f"Could not find {candidates} in: {CFG.input_roots()}")
    return None


print(f"Working directory: {CFG.INPUT_DIR}")
print(f"Output directory: {CFG.RESULTS_DIR}")

prediction_path = resolve_file(CFG.PREDICTION_CANDIDATES)
hard_class_path = resolve_file(CFG.HARD_CLASS_CANDIDATES)
confusion_path = resolve_file(CFG.CONFUSION_CANDIDATES, required=False)
metrics_path = resolve_file((CFG.METRICS_FILE,), required=False)

print(f"Predictions: {prediction_path}")
print(f"Weak class report: {hard_class_path}")
print(f"Confusion report: {confusion_path}")
print(f"Metrics file: {metrics_path}")

raw_predictions = pd.read_csv(prediction_path)
weak_classes_df = pd.read_csv(hard_class_path)
confusion_df = pd.read_csv(confusion_path) if confusion_path else pd.DataFrame(columns=["actual", "predicted", "count"])
metrics_df = pd.read_csv(metrics_path) if metrics_path else pd.DataFrame()

print(f"Predictions rows: {len(raw_predictions):,}")



## 3. Feature Engineering

- normalize E2 schema (`true_label`, `pred_label`, `top_5_labels`),
- build `is_hard_case` and `is_frequent_confusion_pair` flags,
- compute top-1/top-2 margin when possible,
- persist the engineered feature table for review.



In [ ]:
def parse_float(value: str) -> list[float]:
    out = []
    for item in str(value).split("|"):
        item = item.strip()
        if item:
            out.append(float(item))
    return out


def parse_labels(value: str) -> list[str]:
    return [item.strip() for item in str(value).split("|") if item.strip()]


def first_existing_column(df: pd.DataFrame, candidates: tuple[str, ...]) -> str:
    for column in candidates:
        if column in df.columns:
            return column
    raise KeyError(f"Expected one of {candidates}. Available: {df.columns.tolist()}")


def normalize_predictions(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize E2 or legacy prediction schema to a decision-layer baseline."""
    true_col = first_existing_column(df, ("true_label", "actual"))
    pred_col = first_existing_column(df, ("pred_label", "predicted"))
    conf_col = first_existing_column(df, ("confidence", "top_1_prob", "top_1_probability", "test_top_1_probability"))
    top5_col = first_existing_column(df, ("top_5_labels", "top_5"))
    top5_conf_col = "top_5_confidence" if "top_5_confidence" in df.columns else None

    out = df.copy()
    out = out.rename(
        columns={
            true_col: "true_label",
            pred_col: "pred_label",
            conf_col: "top_1_confidence",
            top5_col: "top_5_labels",
        }
    )
    out["top_5_labels"] = out["top_5_labels"].apply(parse_labels)
    out["top_5_confidences"] = (
        out[top5_conf_col].apply(parse_float) if top5_conf_col else [[] for _ in range(len(out))]
    )
    out["top_5_confidences"] = out["top_5_confidences"].apply(
        lambda values: values if len(values) == 5 else (values + [0.0] * (5 - len(values)))
    )
    out["top_1_confidence"] = out["top_1_confidence"].astype(float)
    out["top_2_confidence"] = out["top_5_confidences"].apply(lambda values: values[1])
    out["top_1_top_2_margin"] = out["top_1_confidence"] - out["top_2_confidence"]
    out["actual"] = out["true_label"].astype(str)
    out["predicted"] = out["pred_label"].astype(str)
    if "is_correct" not in out.columns:
        out["is_correct"] = out["pred_label"].astype(str) == out["true_label"].astype(str)
    if "is_top_5_correct" not in out.columns:
        out["is_top_5_correct"] = out.apply(lambda row: row["true_label"] in row["top_5_labels"], axis=1)
    out["top_5_contains_actual"] = out["is_top_5_correct"]
    return out


def weak_class_names(df: pd.DataFrame) -> set[str]:
    for col in ("class_name", "class"):
        if col in df.columns:
            return {str(v) for v in df[col].dropna().tolist()}
    return set()


def derive_confusion_pairs(pred_df: pd.DataFrame, provided_df: pd.DataFrame) -> pd.DataFrame:
    if not provided_df.empty and {"actual", "predicted"}.issubset(provided_df.columns):
        cleaned = provided_df.copy()
        if "count" not in cleaned.columns:
            cleaned["count"] = 1
        return cleaned[["actual", "predicted", "count"]]

    base = pred_df[(pred_df["actual"] != pred_df["predicted"])][["actual", "predicted"]].copy()
    if base.empty:
        return pd.DataFrame(columns=["actual", "predicted", "count"])
    counts = base.value_counts().rename("count").reset_index()
    return counts


def frequent_confusion_set(df: pd.DataFrame) -> set[tuple[str, str]]:
    if df.empty:
        return set()
    top_pairs = df.sort_values("count", ascending=False).head(80)
    return {(str(row.actual), str(row.predicted)) for row in top_pairs.itertuples()}


features_df = normalize_predictions(raw_predictions)
hard_class_names = weak_class_names(weak_classes_df)
confusion_pairs_df = derive_confusion_pairs(features_df[["actual", "predicted"]], confusion_df)
frequent_confusion_pairs = frequent_confusion_set(confusion_pairs_df)

features_df["is_hard_actual_class"] = features_df["actual"].isin(hard_class_names)
features_df["is_hard_predicted_class"] = features_df["predicted"].isin(hard_class_names)
features_df["is_hard_case"] = features_df["is_hard_actual_class"] | features_df["is_hard_predicted_class"]
features_df["is_frequent_confusion_pair"] = features_df.apply(
    lambda row: (row["actual"], row["predicted"]) in frequent_confusion_pairs,
    axis=1,
)

features_df.to_csv(CFG.RESULTS_DIR / "decision_features.csv", index=False)
features_df[[
    "actual",
    "predicted",
    "top_1_confidence",
    "top_1_top_2_margin",
    "is_correct",
    "top_5_contains_actual",
    "is_hard_case",
    "is_frequent_confusion_pair",
]].head()



## 4. Threshold Search

Search constraints and thresholds to maximize auto-accept trust first,
then suggest coverage.



In [ ]:
def assign_decision_band(
    row: pd.Series,
    auto_confidence: float,
    suggest_confidence: float,
    margin_threshold: float,
) -> str:
    """Assign a product decision band to one prediction."""
    if row["is_frequent_confusion_pair"]:
        return "review"
    if row["is_hard_case"] and row["top_1_confidence"] < auto_confidence:
        return "confirm"
    if (
        row["top_1_confidence"] >= auto_confidence
        and row["top_1_top_2_margin"] >= margin_threshold
        and not row["is_hard_case"]
    ):
        return "auto_accept"
    if row["top_1_confidence"] >= suggest_confidence and row["top_5_contains_actual"]:
        return "suggest"
    return "confirm"


def decision_band_metrics(decision_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize coverage and accuracy by decision band."""
    rows = []
    total = len(decision_df)
    for band, band_df in decision_df.groupby("decision_band"):
        rows.append(
            {
                "decision_band": band,
                "sample_count": len(band_df),
                "coverage": len(band_df) / total,
                "top_1_accuracy": band_df["is_correct"].mean(),
                "top_5_contains_actual": band_df["top_5_contains_actual"].mean(),
                "mean_confidence": band_df["top_1_confidence"].mean(),
                "mean_margin": band_df["top_1_top_2_margin"].mean(),
            }
        )
    return pd.DataFrame(rows).sort_values("decision_band")


def evaluate_policy(auto_confidence: float, suggest_confidence: float, margin_threshold: float) -> dict:
    """Evaluate one threshold policy."""
    decision_df = features_df.copy()
    decision_df["decision_band"] = decision_df.apply(
        assign_decision_band,
        axis=1,
        auto_confidence=auto_confidence,
        suggest_confidence=suggest_confidence,
        margin_threshold=margin_threshold,
    )
    metrics_df = decision_band_metrics(decision_df)
    metrics = {
        "auto_confidence": auto_confidence,
        "suggest_confidence": suggest_confidence,
        "margin_threshold": margin_threshold,
    }
    for _, row in metrics_df.iterrows():
        band = row["decision_band"]
        metrics[f"{band}_coverage"] = row["coverage"]
        metrics[f"{band}_top_1_accuracy"] = row["top_1_accuracy"]
        metrics[f"{band}_top_5_contains_actual"] = row["top_5_contains_actual"]
    return metrics


policy_rows = []
for auto_confidence in CFG.AUTO_CONFIDENCE_GRID:
    for suggest_confidence in CFG.SUGGEST_CONFIDENCE_GRID:
        if suggest_confidence >= auto_confidence:
            continue
        for margin_threshold in CFG.MARGIN_GRID:
            policy_rows.append(
                evaluate_policy(auto_confidence, suggest_confidence, margin_threshold)
            )

policy_search_df = pd.DataFrame(policy_rows).fillna(0.0)
policy_search_df["meets_auto_accuracy"] = (
    policy_search_df["auto_accept_top_1_accuracy"] >= CFG.MIN_AUTO_ACCEPT_ACCURACY
)
policy_search_df["meets_suggest_accuracy"] = (
    policy_search_df["suggest_top_5_contains_actual"] >= CFG.MIN_SUGGEST_ACCURACY
)
policy_search_df["policy_score"] = (
    2.0 * policy_search_df["auto_accept_coverage"]
    + policy_search_df["suggest_coverage"]
    - 0.5 * policy_search_df["review_coverage"]
)
eligible_policy_df = policy_search_df[
    policy_search_df["meets_auto_accuracy"]
    & policy_search_df["meets_suggest_accuracy"]
].copy()

if eligible_policy_df.empty:
    best_policy = policy_search_df.sort_values("policy_score", ascending=False).iloc[0]
    print("No policy met all minimum targets; using highest score policy.")
else:
    best_policy = eligible_policy_df.sort_values("policy_score", ascending=False).iloc[0]

policy_search_df.to_csv(CFG.RESULTS_DIR / "decision_policy_search.csv", index=False)
best_policy.to_frame().T



## 5. Final Policy and Outputs

Export policy tables and per-band examples for downstream demo/backend usage.



In [ ]:
AUTO_CONFIDENCE = float(best_policy["auto_confidence"])
SUGGEST_CONFIDENCE = float(best_policy["suggest_confidence"])
MARGIN_THRESHOLD = float(best_policy["margin_threshold"])

decision_df = features_df.copy()
decision_df["decision_band"] = decision_df.apply(
    assign_decision_band,
    axis=1,
    auto_confidence=AUTO_CONFIDENCE,
    suggest_confidence=SUGGEST_CONFIDENCE,
    margin_threshold=MARGIN_THRESHOLD,
)

band_metrics_df = decision_band_metrics(decision_df)
policy_df = pd.DataFrame(
    [
        {
            "auto_confidence": AUTO_CONFIDENCE,
            "suggest_confidence": SUGGEST_CONFIDENCE,
            "margin_threshold": MARGIN_THRESHOLD,
            "min_auto_accept_accuracy": CFG.MIN_AUTO_ACCEPT_ACCURACY,
            "min_suggest_accuracy": CFG.MIN_SUGGEST_ACCURACY,
            "e2_test_top_1_accuracy": float(
                metrics_df.loc[metrics_df["split"] == "test", "top_1_accuracy"].iloc[0]
            )
            if not metrics_df.empty and "split" in metrics_df.columns
            else np.nan,
            "e2_test_top_5_accuracy": float(
                metrics_df.loc[metrics_df["split"] == "test", "top_5_accuracy"].iloc[0]
            )
            if not metrics_df.empty and "split" in metrics_df.columns
            else np.nan,
            "e2_test_ece_calibrated": float(
                metrics_df.loc[metrics_df["split"] == "test", "ece_calibrated"].iloc[0]
            )
            if not metrics_df.empty and "split" in metrics_df.columns
            else np.nan,
        }
    ]
)

policy_search_df.to_csv(CFG.RESULTS_DIR / "decision_policy_search.csv", index=False)
decision_df.to_csv(CFG.RESULTS_DIR / "test_predictions_with_decisions.csv", index=False)
band_metrics_df.to_csv(CFG.RESULTS_DIR / "decision_band_metrics.csv", index=False)
policy_df.to_csv(CFG.RESULTS_DIR / "decision_policy.csv", index=False)

for band in ["auto_accept", "suggest", "confirm", "review"]:
    examples = decision_df[decision_df["decision_band"] == band].head(25)
    examples.to_csv(CFG.RESULTS_DIR / f"decision_examples_{band}.csv", index=False)

confusion_pairs_df.to_csv(CFG.RESULTS_DIR / "top_confusion_pairs.csv", index=False)
weak_classes_df.to_csv(CFG.RESULTS_DIR / "weak_classes.csv", index=False)

# App-facing payloads
(CFG.RESULTS_DIR / "decision_policy.json").write_text(
    policy_df.to_json(orient="records", indent=2)
)
(CFG.RESULTS_DIR / "hard_classes.json").write_text(
    json.dumps(sorted(hard_class_names), indent=2)
)
(CFG.RESULTS_DIR / "top_confusion_pairs.json").write_text(
    json.dumps(
        confusion_pairs_df.to_dict(orient="records") if not confusion_pairs_df.empty else [],
        indent=2,
    )
)

# Bundle outputs for easy download.
artifact_zip = CFG.RESULTS_DIR / CFG.ZIP_FILE_NAME
with zipfile.ZipFile(artifact_zip, "w") as zf:
    for artifact in [
        CFG.RESULTS_DIR / "decision_features.csv",
        CFG.RESULTS_DIR / "decision_policy_search.csv",
        CFG.RESULTS_DIR / "test_predictions_with_decisions.csv",
        CFG.RESULTS_DIR / "decision_band_metrics.csv",
        CFG.RESULTS_DIR / "decision_policy.csv",
        CFG.RESULTS_DIR / "top_confusion_pairs.csv",
        CFG.RESULTS_DIR / "weak_classes.csv",
        CFG.RESULTS_DIR / "decision_policy.json",
        CFG.RESULTS_DIR / "hard_classes.json",
        CFG.RESULTS_DIR / "top_confusion_pairs.json",
    ]:
        if artifact.exists():
            zf.write(artifact, arcname=artifact.name)

print("Selected policy")
display(policy_df)
print("Band metrics")
display(band_metrics_df)
print(f"Saved policy bundle: {artifact_zip}")



## 6. Decision Helper

Use this helper with one normalized prediction row and get the band result.



In [ ]:
def recommend_action(prediction_row: pd.Series) -> str:
    """Return the decision action for one prediction row."""
    row = prediction_row.copy()
    if "actual" not in row and "true_label" in row:
        row["actual"] = row["true_label"]
    if "predicted" not in row and "pred_label" in row:
        row["predicted"] = row["pred_label"]
    if isinstance(row.get("top_5_labels"), str):
        row["top_5_labels"] = parse_labels(row["top_5_labels"])
    row["top_5_confidences"] = parse_float(row.get("top_5_confidence", "")) if row.get("top_5_confidence") else []
    if "top_1_confidence" not in row:
        row["top_1_confidence"] = float(row.get("confidence", 0.0))
    row["top_2_confidence"] = row["top_5_confidences"][1] if len(row["top_5_confidences"]) > 1 else 0.0
    row["top_1_top_2_margin"] = row["top_1_confidence"] - row["top_2_confidence"]
    row["is_hard_case"] = str(row["actual"]) in hard_class_names or str(row["predicted"]) in hard_class_names
    row["is_frequent_confusion_pair"] = (str(row["actual"]), str(row["predicted"])) in frequent_confusion_pairs
    row["top_5_contains_actual"] = str(row["actual"]) in row["top_5_labels"]
    return assign_decision_band(
        row,
        auto_confidence=AUTO_CONFIDENCE,
        suggest_confidence=SUGGEST_CONFIDENCE,
        margin_threshold=MARGIN_THRESHOLD,
    )

sample = raw_predictions.iloc[0].copy()
print(f"Recommended action: {recommend_action(sample)}")



## 7. Run Insight

- `auto_accept` is for high-confidence and non-risky classes.
- `suggest` is for useful ranked alternatives.
- `confirm` / `review` are intentional safety gates for weak or repeated confusion cases.

The notebook now produces a reusable policy bundle for the 130-class path.

